## udf()
A User Defined Function (UDF) allows you to define custom functions in Python, Scala, or SQL and use them in Spark DataFrames. UDFs are useful when built-in functions do not meet your requirements.

**Example (Python):**
python
from pyspark.sql.functions import udf
from pyspark.sql.types import StringType

def upper_case(s):
    return s.upper() if s else None

upper_case_udf = udf(upper_case, StringType())

df.withColumn("upper_name", upper_case_udf(df["name"]))


**Note:** UDFs can be slower than native Spark functions. Use built-in functions when possible.

In [0]:
data = [(1,"raj",5000,200),(2,"ravi",7000,500),(3,"sruthi",10000,800)]
schema = ["id", "name", "salary", "bonus"]
df = spark.createDataFrame(data, schema)
df.show()

+---+------+------+-----+
| id|  name|salary|bonus|
+---+------+------+-----+
|  1|   raj|  5000|  200|
|  2|  ravi|  7000|  500|
|  3|sruthi| 10000|  800|
+---+------+------+-----+



In [0]:
from pyspark.sql.functions import udf
from pyspark.sql.types import *
def totalpay(s,b):
    return s+b
totalPayment = udf(lambda s,b:totalpay(s,b),IntegerType())



In [0]:
df.withColumn("totalpay",totalPayment(df.salary,df.bonus)).show()


+---+------+------+-----+--------+
| id|  name|salary|bonus|totalpay|
+---+------+------+-----+--------+
|  1|   raj|  5000|  200|    5200|
|  2|  ravi|  7000|  500|    7500|
|  3|sruthi| 10000|  800|   10800|
+---+------+------+-----+--------+



In [0]:
@udf(returnType=IntegerType())
def doublepay(s,b):
    return 2*(s+b)
df.select("*",doublepay(df.salary,df.bonus).alias("doublepay")).show()

+---+------+------+-----+---------+
| id|  name|salary|bonus|doublepay|
+---+------+------+-----+---------+
|  1|   raj|  5000|  200|    10400|
|  2|  ravi|  7000|  500|    15000|
|  3|sruthi| 10000|  800|    21600|
+---+------+------+-----+---------+



### Using UDF in SQL

You can register a Python UDF and use it in SQL queries in Databricks.

**Example:**

python
from pyspark.sql.types import IntegerType
from pyspark.sql.functions import udf

@udf(returnType=IntegerType())
def totalpay(s, b):
    return s + b

spark.udf.register("TotalPay", totalpay)


Now you can use the UDF in SQL:

sql
SELECT *, TotalPay(salary, bonus) AS totalpay
FROM df

In [0]:
data = [(1,"raj",5000,200),(2,"ravi",7000,500),(3,"sruthi",10000,800)]
schema = ["id", "name", "salary", "bonus"]
df = spark.createDataFrame(data, schema)
df.show()
df.createOrReplaceTempView("emps")

+---+------+------+-----+
| id|  name|salary|bonus|
+---+------+------+-----+
|  1|   raj|  5000|  200|
|  2|  ravi|  7000|  500|
|  3|sruthi| 10000|  800|
+---+------+------+-----+



In [0]:
from pyspark.sql.functions import udf
from pyspark.sql.types import *
def totalpay(s,b):
    return s+b
totalPayment = udf(lambda s,b:totalpay(s,b),IntegerType())


In [0]:
spark.udf.register(name="Totalpay",f=totalpay,returnType=IntegerType())

<function __main__.totalpay(s, b)>

In [0]:
%sql
select *,TotalPay(salary,bonus) as totpay
from emps

id,name,salary,bonus,totpay
1,raj,5000,200,5200
2,ravi,7000,500,7500
3,sruthi,10000,800,10800


## RDD
An RDD (Resilient Distributed Dataset) is the fundamental data structure of Apache Spark. It represents an immutable, distributed collection of objects that can be processed in parallel. RDDs support fault tolerance, transformations (like `map`, `filter`), and actions (like `collect`, `count`).

**Example (Python):**
python
rdd = spark.sparkContext.parallelize([1, 2, 3, 4])
rdd.map(lambda x: x * 2).collect()

In [0]:
data = [(1,"raj"),(2,"ravi"),(3,"sruthi")]
schema = ["id", "name"]
rdd = spark.sparkContext.parallelize(data)
df = spark.createDataFrame(rdd,schema)
df.show()


---------------------------------------------------------------------------
PySparkAttributeError                     Traceback (most recent call last)
File <command-5237962295004647>, line 3
      1 data = [(1,"raj"),(2,"ravi"),(3,"sruthi")]
      2 schema = ["id", "name"]
----> 3 rdd = spark.sparkContext.parallelize(data)
      4 df = spark.createDataFrame(rdd,schema)
      5 df.show()

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/session.py:1130, in SparkSession.__getattr__(self, name)
   1128 def __getattr__(self, name: str) -> Any:
   1129     if name in ["_jsc", "_jconf", "_jvm", "_jsparkSession", "sparkContext", "newSession"]:
-> 1130         raise PySparkAttributeError(
   1131             errorClass="JVM_ATTRIBUTE_NOT_SUPPORTED", messageParameters={"attr_name": name}
   1132         )
   1133     return object.__getattribute__(self, name)

PySparkAttributeError: [JVM_ATTRIBUTE_NOT_SUPPORTED] SparkContext is not supported on serverless compute. If yo